# Exploratory Data Analysis and Data Wrangling

This notebook walks through the first — and in practice the longest — part of every machine learning project: getting to know the data, cleaning and validating it, finding its traps, and turning what we learn into features and modelling decisions. We work with a real banking dataset that contains most of the traps you will meet in practice.

Author: Michał Woźniak

## Technical Setup

This notebook is part of the course repository, which already ships a complete, locked environment (`pyproject.toml` + `uv.lock`). You do **not** need to install packages one by one.

```bash
# from the root of the repository
uv sync
uv run jupyter lab
```

A detailed, step-by-step installation guide (what `uv` is, how to install it on Windows / macOS / Linux, how to select the kernel in VS Code, troubleshooting) is available in [`SETUP.md`](../SETUP.md). If you need to refresh your Python, NumPy or pandas, work through [`python_course.ipynb`](python_course.ipynb) first.

**Packages used in this notebook:** `numpy`, `pandas`, `matplotlib`, `seaborn`, `scipy`, `scikit-learn`, `statsmodels`.

**Data:** the dataset ships with the repository (`data/bank_marketing.csv.gz`) — no internet access is needed.

# 1. Introduction

## The Business Problem

A Portuguese retail bank runs **telemarketing campaigns** to sell term deposits. Each call costs money and annoys clients who are not interested, so the bank wants to call only those who are likely to subscribe.

- **Unit of observation:** one phone contact with a client (41,188 contacts, May 2008 – November 2010)
- **Target:** `y` — did the client subscribe to a term deposit after this contact? (binary classification)
- **Features:** client data, information about the current and previous campaigns, and the macro-economic context at the time of the call

Source: Moro, S., Cortez, P., Rita, P. (2014). *A Data-Driven Approach to Predict the Success of Bank Telemarketing*. Decision Support Systems, 62, 22–31 (UCI Machine Learning Repository, CC BY 4.0 — see [`data/README.md`](../data/README.md)).

## Data Dictionary

| Group | Variable | Description |
|---|---|---|
| Client | `age` | age in years |
| | `job`, `marital`, `education` | type of job, marital status, education level |
| | `default`, `housing`, `loan` | has credit in default? / a housing loan? / a personal loan? |
| Current campaign | `contact` | communication type (cellular / telephone) |
| | `month`, `day_of_week` | month and day of the week of the last contact |
| | `duration` | duration of the last contact in seconds |
| | `campaign` | number of contacts with this client during this campaign (including this one) |
| Previous campaigns | `pdays` | days since the client was last contacted in a previous campaign (**999 = not previously contacted**) |
| | `previous` | number of contacts with this client before this campaign |
| | `poutcome` | outcome of the previous campaign (failure / nonexistent / success) |
| Macro context | `emp.var.rate` | employment variation rate (quarterly) |
| | `cons.price.idx`, `cons.conf.idx` | consumer price index, consumer confidence index (monthly) |
| | `euribor3m` | 3-month Euribor rate (daily) |
| | `nr.employed` | number of employees in the economy, thousands (quarterly) |
| Target | `y` | has the client subscribed to a term deposit? (yes / no) |

The documentation also states: *"missing values in some categorical attributes are coded with the `unknown` label"* and *"the examples are ordered by date"*. Both remarks will turn out to be crucial.

## Dataset Preparation Steps

The slides shown at the start of this lab (Chapter 4: *Dataset preparation steps*) list the typical steps of data preparation. This is where each of them appears in the notebook:

| Step | Section |
|---|---|
| Data ingestion, transformation to a convenient analytical form | 2 |
| Initial data exploration and validation | 2, 3 |
| Data cleaning (duplicates, hidden missing values, sentinel values, data types) | 3 |
| Detecting target leakage | 4 |
| Division into training and test sets | 5 |
| Extensive EDA — univariate, bivariate, multivariate, with visualisations and statistics | 6, 7, 8 |
| Initial feature selection | 7.3, 8.2 |
| Stability of the features over time | 9 |
| Feature engineering | 10.1 |
| Saving the data in an efficient format | 10.2 |
| Turning the findings into modelling decisions | 11 |

> **The golden rule of this notebook:** everything that *learns something from the data* (imputation values, scaling parameters, category statistics, feature rankings, ...) must be learned on the **training set only**. We therefore split the data *before* analysing relationships with the target.

**What this notebook deliberately does not do.** It fits no models: its output is clean data, an understanding of it, and a list of decisions for the modelling stage. The tools that come next have their own notebooks:

| Topic | Where |
|---|---|
| Linear and logistic regression (scikit-learn and `statsmodels`, coefficients, odds ratios, regularisation) | `linear_and_logistic_regression.ipynb` |
| Imputation strategies, scaling, encoders, binning, interactions as scikit-learn transformers | `core_ml_techniques.ipynb`, Sections 1–2 |
| Algorithmic feature selection (filter, wrapper and embedded methods) | `core_ml_techniques.ipynb`, Section 4 |
| Drift detection and monitoring a deployed model | `core_ml_techniques.ipynb`, Section 8 |
| Cross-validation schemes for time-ordered data (`TimeSeriesSplit`) | `core_ml_techniques.ipynb`, Section 10 |

# 2. Data Ingestion and First Look

In [ ]:
# Import necessary libraries
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Matplotlib configuration for better plots
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

# Show all columns of wide data frames
pd.set_option("display.max_columns", 50)

print("Libraries imported successfully!")

`read_csv` has dozens of arguments, and real files rarely work with the defaults. Here the separator is a semicolon (common in European exports, where the comma is the decimal mark) and the file is gzip-compressed — pandas decompresses it on the fly.

**Always look at the raw file first** (with a text editor, or `head` in the terminal): separator, decimal mark, header, encoding, how missing values are written.

In [ ]:
DATA_PATH = Path("..") / "data" / "bank_marketing.csv.gz"

df_raw = pd.read_csv(DATA_PATH, sep=";")

print(f"Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Memory usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

df_raw.head()

In [ ]:
# Column names, non-null counts and data types
df_raw.info()

`info()` reports **no missing values** — every column has 41,188 non-null entries. Do not trust it yet: it only counts values that pandas *recognises* as missing (`NaN`). We come back to this in Section 3.3.

Next, summary statistics. For numerical columns look at the **range** (are the minimum and the maximum plausible?), and at the **mean vs. the median** (a large gap indicates skewness or outliers).

In [ ]:
df_raw.describe().T.round(2)

In [ ]:
# Categorical columns: number of distinct values, the most frequent one and its share
categorical_summary = df_raw.describe(include="str").T
categorical_summary["top_share"] = (categorical_summary["freq"] / categorical_summary["count"]).astype(float).round(3)
categorical_summary

**First observations — a list of things to investigate:**

- `pdays`: the median *and* the 75th percentile equal **999** — the "not previously contacted" code dominates the column and would wreck any statistic or distance computed from it.
- `duration` ranges from 0 to almost 5,000 seconds; `campaign` goes up to 56 contacts with one client — long right tails.
- `default`: the most frequent value covers ~79% — we need to check what the rest is.
- The macro variables have very few distinct values relative to 41,188 rows — they describe the *moment* of the call, not the client.
- The target is **imbalanced** (the most frequent value, `no`, covers ~89%).

# 3. Data Cleaning and Validation

## 3.1 Column Names

Dots in column names are legal but inconvenient: `df.emp.var.rate` does not work, and formula interfaces (e.g. in `statsmodels`) and many other tools interpret them as operators. We replace them with underscores and give the target a descriptive name.

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.replace(".", "_", regex=False)
df = df.rename(columns={"y": "subscribed"})

print(df.columns.tolist())

## 3.2 Duplicates

A fully duplicated row can be a genuine coincidence or a technical artefact (a double export, a join gone wrong). With 20 features — including the call duration in seconds — a coincidence is very unlikely.

In [ ]:
n_duplicates = df.duplicated().sum()
print(f"Fully duplicated rows: {n_duplicates}")

# keep=False marks ALL copies, so that we can look at the pairs
df[df.duplicated(keep=False)].sort_values(["age", "duration"]).head(6)

In [ ]:
# Drop the duplicates and reset the index (the ORDER of rows matters in this dataset - see Section 3.5)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after removing duplicates: {df.shape}")

## 3.3 Hidden Missing Values

Missing values are rarely stored as `NaN` in source systems. Typical disguises: `"unknown"`, `"n/a"`, `"?"`, empty strings, `-1`, `0`, `999`, `9999`, `1900-01-01`. The documentation told us about two of them.

### The `"unknown"` label

In [ ]:
categorical_columns = df.select_dtypes(include="str").columns.drop("subscribed")

unknown_table = pd.DataFrame(
    {
        "n_unknown": (df[categorical_columns] == "unknown").sum(),
        "share_unknown": (df[categorical_columns] == "unknown").mean().round(4),
    }
).sort_values("n_unknown", ascending=False)

print(unknown_table[unknown_table["n_unknown"] > 0])

Should `"unknown"` be converted to `NaN`? It depends on **why** the value is missing (the MCAR / MAR / MNAR distinction from the *Data imputation* slide). A quick diagnostic: compare the target rate of the clients with a missing value to the rest. If it differs, the missingness itself carries information, and replacing it with the mode would destroy that information.

In [ ]:
rows = []
for column in unknown_table.index[unknown_table["n_unknown"] > 0]:
    is_unknown = df[column] == "unknown"
    rows.append(
        {
            "variable": column,
            "subscription rate if unknown": (df.loc[is_unknown, "subscribed"] == "yes").mean(),
            "subscription rate if known": (df.loc[~is_unknown, "subscribed"] == "yes").mean(),
        }
    )
print(pd.DataFrame(rows).set_index("variable").round(4))

print("\nValue counts of 'default':")
print(df["default"].value_counts())

**What to notice:**
- For `default`, clients with an unknown status subscribe **much less often** than clients known to have no default. The missingness is informative (the bank probably knows less about clients it has a weaker relationship with), so `"unknown"` deserves to stay as **its own category**.
- `default == "yes"` occurs only 3 times in more than 41,000 rows. In practice this variable is a "known / unknown" flag, not a default indicator — a good example of why you should always look at value counts before interpreting a variable by its name.
- For the other variables the rates are close, and the shares of unknowns are small.

Our decision: keep `"unknown"` as a separate category for all categorical variables. It is simple, loses no information, and needs no statistics learned from the data. (Alternatives — mode imputation, model-based imputation — are covered in `core_ml_techniques.ipynb`.)

### The `999` sentinel in `pdays`

In [ ]:
print(f"Share of pdays == 999: {(df['pdays'] == 999).mean():.2%}")
print("\nDistribution of pdays for previously contacted clients:")
print(df.loc[df["pdays"] != 999, "pdays"].describe().round(2))

# What would a careless analyst conclude?
print(f"\nMean of pdays including the sentinel: {df['pdays'].mean():.1f} days")
print(f"Mean of pdays excluding the sentinel: {df.loc[df['pdays'] != 999, 'pdays'].mean():.1f} days")

A sentinel value inside a numerical column is dangerous: a linear model would read 999 as "a very long time ago", a distance-based model would place these clients far away from everybody else, and every summary statistic is meaningless.

The clean representation uses **two** columns: an indicator (*was the client contacted before?*) and the number of days, which is missing when there was no earlier contact.

In [ ]:
df["pdays"] = df["pdays"].replace(999, np.nan)
df["contacted_before"] = df["pdays"].notna().astype(int)

print(df[["pdays", "contacted_before"]].describe().T.round(2))

## 3.4 Consistency Checks

Validate the data against the **business logic** described in the documentation. Three columns describe previous campaigns, so they must agree with each other:

- `previous == 0` (never contacted before) ⇔ `poutcome == "nonexistent"`
- `pdays` missing (not previously contacted) ⇔ `previous == 0`

In [ ]:
check_1 = ((df["previous"] == 0) != (df["poutcome"] == "nonexistent")).sum()
check_2 = (df["pdays"].isna() != (df["previous"] == 0)).sum()

print(f"Rows violating 'previous == 0  <=>  poutcome is nonexistent': {check_1:,}")
print(f"Rows violating 'pdays missing  <=>  previous == 0':           {check_2:,}")

print("\nCross-tabulation of 'pdays is missing' and 'poutcome':")
print(pd.crosstab(df["pdays"].isna().rename("pdays missing"), df["poutcome"]))

**What to notice:** the first rule holds perfectly, the second one is violated by about four thousand rows. All of them are clients whose previous campaign ended in **failure** — so they *were* contacted before — and yet `pdays` carried the "not previously contacted" code.

The documentation is therefore incomplete: 999 evidently also means "contacted before, but the number of days is not known" (or exceeds some cap). Consequences:

- our new `contacted_before` flag, derived from `pdays`, would be **wrong** for these clients — we must derive it from `previous` instead,
- `pdays` is observed for very few clients, so its practical value is limited.

This is a typical outcome of a consistency check: the data is not "wrong", but it means something different from what the documentation says. In a real project you would now talk to the data owner.

In [ ]:
df["contacted_before"] = (df["previous"] > 0).astype(int)

print(df["contacted_before"].value_counts())
print(f"\nShare of clients with a known pdays value: {df['pdays'].notna().mean():.2%}")

## 3.5 Reconstructing the Time Dimension

The file contains the month and the day of the week of each call, but **no year and no date**. The documentation says that the rows are ordered by date, from May 2008 to November 2010. That is enough to reconstruct the year: walking through the file, every time the month number *decreases* (e.g. from December to March) a new year must have started.

In [ ]:
month_numbers = {
    m: i + 1 for i, m in enumerate(["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"])
}
month_number = df["month"].map(month_numbers)

# A decrease of the month number between consecutive rows marks the start of a new year
new_year_started = month_number.diff() < 0
df["year"] = 2008 + new_year_started.cumsum()
df["year_month"] = pd.to_datetime({"year": df["year"], "month": month_number, "day": 1})

print(f"Number of year changes detected: {new_year_started.sum()}")
print(f"Reconstructed period: {df['year_month'].min():%B %Y} - {df['year_month'].max():%B %Y}")
print("\nContacts per year:")
print(df["year"].value_counts().sort_index())

Exactly two year changes, and the reconstructed period matches the documentation (May 2008 – November 2010) — a strong sign that the reconstruction is right. We can cross-check it with an external fact: the 3-month Euribor stood at about 5% in the autumn of 2008 and collapsed to below 1% during 2009 as the ECB reacted to the financial crisis.

In [ ]:
monthly = df.groupby("year_month").agg(
    n_contacts=("subscribed", "size"),
    subscription_rate=("subscribed", lambda s: (s == "yes").mean()),
    euribor3m=("euribor3m", "mean"),
)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].bar(monthly.index, monthly["n_contacts"], width=20)
axes[0].set_ylabel("Number of contacts", fontsize=12)
axes[0].set_title("The Dataset over Time", fontsize=14, fontweight="bold")

axes[1].plot(monthly.index, monthly["subscription_rate"], "o-", linewidth=2)
axes[1].set_ylabel("Subscription rate", fontsize=12)

axes[2].plot(monthly.index, monthly["euribor3m"], "o-", linewidth=2, color="darkred")
axes[2].set_ylabel("Euribor 3M (%)", fontsize=12)
axes[2].set_xlabel("Month of contact", fontsize=12)

plt.tight_layout()
plt.show()

print(
    monthly.groupby(monthly.index.year)
    .agg(
        n_contacts=("n_contacts", "sum"),
        mean_monthly_subscription_rate=("subscription_rate", "mean"),
        mean_euribor3m=("euribor3m", "mean"),
    )
    .round(3)
)

**What to notice — this plot changes how the whole project has to be approached:**

- The Euribor path reproduces the 2008–2009 collapse of interest rates, which confirms our reconstruction of the dates.
- The dataset is **not a homogeneous sample**. In 2008 the bank made tens of thousands of calls with a subscription rate of a few percent. From 2009 on it made far fewer calls, and around **half** of them were successful. The campaign strategy changed (far more selective targeting) and so did the economy.
- The macro-economic variables are almost constant within a month, so for a model they act largely as a **proxy for calendar time**. A model can use them to learn "late period ⇒ high success rate" — which predicts well inside this dataset, and may mean nothing for a campaign run next year.
- This is **drift** seen from the data side (detecting and monitoring it in a deployed model: `core_ml_techniques.ipynb`, Section 8), and it decides how the model must be validated: see Section 5.

## 3.6 Data Types

Proper data types make the data frame smaller and faster, and they document the meaning of the columns:

- the target becomes an integer 0/1,
- text columns become `category` — **ordered** where the categories have a natural order (education), so that comparisons and sorting work.

In [ ]:
memory_before = df.memory_usage(deep=True).sum() / 1024**2

# Target as 0/1
df["subscribed"] = (df["subscribed"] == "yes").astype(int)

# Ordered categorical: education ('unknown' is placed first; it has no natural position)
education_order = [
    "unknown",
    "illiterate",
    "basic.4y",
    "basic.6y",
    "basic.9y",
    "high.school",
    "professional.course",
    "university.degree",
]
df["education"] = pd.Categorical(df["education"], categories=education_order, ordered=True)

# Calendar variables with a natural order
df["month"] = pd.Categorical(df["month"], categories=list(month_numbers), ordered=True)
df["day_of_week"] = pd.Categorical(df["day_of_week"], categories=["mon", "tue", "wed", "thu", "fri"], ordered=True)

# Remaining text columns -> unordered categories
for column in df.select_dtypes(include="str").columns:
    df[column] = df[column].astype("category")

memory_after = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memory usage: {memory_before:.1f} MB -> {memory_after:.1f} MB")
print()
print(df.dtypes)

# 4. Target Leakage

**Target leakage** means that a feature contains information that would not be available at the moment the prediction has to be made — typically because it is recorded *after* (or *as a consequence of*) the outcome. A leaky feature produces a spectacular model that is useless in production.

The documentation warns about `duration`: *"this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed."* We want to decide **whom to call**, so at prediction time the call has not happened yet.

Let's measure how strong the leak is, using the ROC AUC of the raw variable as a one-feature "model".

In [ ]:
from sklearn.metrics import roc_auc_score

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.boxplot(data=df, x="subscribed", y="duration", ax=axes[0], showfliers=False)
axes[0].set_title("Call Duration by Outcome (outliers hidden)", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Subscribed", fontsize=12)
axes[0].set_ylabel("Duration (seconds)", fontsize=12)

duration_bins = pd.cut(df["duration"], bins=[-1, 60, 120, 180, 300, 600, 900, 5000])
df.groupby(duration_bins, observed=True)["subscribed"].mean().plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Subscription Rate by Call Duration", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Duration (seconds)", fontsize=12)
axes[1].set_ylabel("Subscription rate", fontsize=12)
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

print(f"ROC AUC of 'duration' alone: {roc_auc_score(df['subscribed'], df['duration']):.4f}")
print(
    f"Calls with duration == 0: {(df['duration'] == 0).sum()}, of which subscribed: {df.loc[df['duration'] == 0, 'subscribed'].sum()}"
)

A single variable reaches an AUC that many full models never achieve — because a long call is largely a *consequence* of the client being interested. We drop it.

**A checklist for spotting leakage:**

1. For every feature ask: *at the moment of prediction, would I already know this value?*
2. Be suspicious of any feature with an implausibly high univariate predictive power.
3. Look for variables updated after the event (account status after default, "date closed", number of reminders sent, ...).
4. Be careful with identifiers and row order — they may encode time or the data collection process.
5. Remember the indirect form: statistics (means, encodings, scalers) computed on the **full** dataset leak information from the test set into training.

In [ ]:
df = df.drop(columns="duration")
print(f"Remaining columns ({df.shape[1]}): {df.columns.tolist()}")

# 5. Train / Test Split

We split **before** analysing the relationships between features and target, so that the test set plays no part in any modelling decision — including the informal decisions we make by looking at plots.

How should we split? Section 3.5 showed that the data is ordered in time and that the world changed during the sample. The model will be used on **future** campaigns, so the honest question is: *how well does a model trained on the past predict the future?* That calls for an **out-of-time split**: train on the earliest 80% of the contacts, test on the latest 20%.

Let's compare it with the usual random (stratified) split.

In [ ]:
from sklearn.model_selection import train_test_split

# Out-of-time split: the rows are ordered by date
cutoff = int(len(df) * 0.8)
train, test = df.iloc[:cutoff].copy(), df.iloc[cutoff:].copy()

# Random stratified split - for comparison only
train_random, test_random = train_test_split(df, test_size=0.2, random_state=42, stratify=df["subscribed"])

summary = pd.DataFrame(
    {
        "rows": [len(train), len(test), len(train_random), len(test_random)],
        "first month": [
            part["year_month"].min().strftime("%Y-%m") for part in (train, test, train_random, test_random)
        ],
        "last month": [part["year_month"].max().strftime("%Y-%m") for part in (train, test, train_random, test_random)],
        "subscription rate": [part["subscribed"].mean() for part in (train, test, train_random, test_random)],
        "mean euribor3m": [part["euribor3m"].mean() for part in (train, test, train_random, test_random)],
    },
    index=["out-of-time: train", "out-of-time: test", "random: train", "random: test"],
)
print(summary.round(3))

**What to notice:**
- With the random split, training and test sets look identical — each test observation has "neighbours" from the same month in the training set. The evaluation answers the question *"how well can I fill in gaps in the past?"*
- With the out-of-time split the two sets differ dramatically: the test period has a several times higher subscription rate and a completely different interest-rate environment. This is the situation the deployed model would really have faced.
- Neither split is "wrong" — they answer different questions. For anything that will be used for **forecasting**, the out-of-time split is the honest one. Section 9 measures how strongly every feature shifts between the two periods; the matching cross-validation scheme (`TimeSeriesSplit`) is covered in `core_ml_techniques.ipynb`, Section 10.

**From now on, all exploration uses the training set only.**

# 6. Univariate Analysis

One variable at a time: what does its distribution look like? Are there outliers, rare categories, implausible values?

## 6.1 Numerical Variables

In [ ]:
from scipy import stats

numerical_columns = [
    "age",
    "campaign",
    "pdays",
    "previous",
    "emp_var_rate",
    "cons_price_idx",
    "cons_conf_idx",
    "euribor3m",
    "nr_employed",
]

numerical_summary = train[numerical_columns].describe().T
numerical_summary["missing"] = train[numerical_columns].isna().mean()
numerical_summary["n_unique"] = train[numerical_columns].nunique()
numerical_summary["skewness"] = train[numerical_columns].skew()
numerical_summary["excess kurtosis"] = train[numerical_columns].kurt()
print(numerical_summary.round(2))

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
for ax, column in zip(axes.ravel(), numerical_columns):
    sns.histplot(train[column].dropna(), bins=40, ax=ax)
    ax.set_title(f"{column} (skewness: {train[column].skew():.2f})", fontsize=12, fontweight="bold")
    ax.set_xlabel("")
plt.suptitle("Distributions of the Numerical Variables (training set)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

**What to notice:**
- `age` is roughly bell-shaped with a right tail; `campaign` and `previous` are **counts** with extreme right skewness — a log transformation or binning will help linear models.
- The macro variables are not "distributions" in the usual sense: they take only a handful of values, one per month or quarter.
- `pdays` is observed for a tiny fraction of the training set.

### Outliers

A common rule flags observations further than 1.5 × IQR from the quartiles. It is only a **screening device**: whether a flagged value is an error, or a rare but genuine observation, is a business question.

In [ ]:
def iqr_outlier_share(series):
    """Share of observations outside [Q1 - 1.5 * IQR, Q3 + 1.5 * IQR]."""
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).mean()


print("Share of IQR outliers:")
for column in ["age", "campaign"]:
    print(f"  {column:10s} {iqr_outlier_share(train[column]):.2%}")

print("\nThe most contacted clients in the training set:")
print(train["campaign"].value_counts().sort_index().tail(5))

print(f"\nSubscription rate when campaign <= 3:  {train.loc[train['campaign'] <= 3, 'subscribed'].mean():.4f}")
print(f"Subscription rate when campaign >= 10: {train.loc[train['campaign'] >= 10, 'subscribed'].mean():.4f}")

Calling the same client dozens of times is not a data error — it is a (questionable) business practice, and these observations are informative: persistence clearly does not pay off. We keep them, and tame the tail with a transformation instead of deleting rows (Section 9).

## 6.2 Categorical Variables

In [ ]:
categorical_columns = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]

fig, axes = plt.subplots(2, 5, figsize=(24, 9))
for ax, column in zip(axes.ravel(), categorical_columns):
    shares = train[column].value_counts(normalize=True, sort=not train[column].cat.ordered)
    shares.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title(column, fontsize=13, fontweight="bold")
    ax.set_xlabel("Share")
    ax.set_ylabel("")
plt.suptitle("Distributions of the Categorical Variables (training set)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# Rare categories are a practical problem: unstable estimates, and they may be absent from a CV fold or the test set
print("Categories with fewer than 1% of the training observations:")
for column in categorical_columns:
    shares = train[column].value_counts(normalize=True)
    rare = shares[(shares < 0.01) & (shares > 0)]
    if len(rare) > 0:
        print(f"  {column:12s} {dict(rare.round(4))}")

# 7. Bivariate Analysis: Features vs. Target

Which variables are related to the target, how strongly, and in what *form* (linear? monotone? U-shaped?). The right tool depends on the types of the two variables:

| Feature | Target | Visualisation | Measure of association | Test |
|---|---|---|---|---|
| numerical | binary | box / violin plot, target rate by bins | point-biserial correlation, AUC of the raw variable | t-test, Mann–Whitney U |
| categorical | binary | target rate by category | Cramér's V, Information Value | χ² test of independence |
| numerical | numerical | scatter plot | Pearson, Spearman | correlation test |
| categorical | categorical | heatmap of the contingency table | Cramér's V | χ² test |
| numerical | categorical | box plot by category | correlation ratio η² | ANOVA, Kruskal–Wallis |

> **A warning about p-values.** With tens of thousands of observations almost *every* difference is statistically significant. Significance tells you that an effect is not zero; it does not tell you that it is large enough to matter. Always look at the **effect size** as well.

## 7.1 Numerical Features vs. Target

In [ ]:
bivariate_numerical = []
for column in numerical_columns:
    observed = train[[column, "subscribed"]].dropna()
    group_0 = observed.loc[observed["subscribed"] == 0, column]
    group_1 = observed.loc[observed["subscribed"] == 1, column]
    auc = roc_auc_score(observed["subscribed"], observed[column])
    bivariate_numerical.append(
        {
            "variable": column,
            "n": len(observed),
            "mean (y=0)": group_0.mean(),
            "mean (y=1)": group_1.mean(),
            "point-biserial r": stats.pointbiserialr(observed["subscribed"], observed[column])[0],
            "Mann-Whitney p": stats.mannwhitneyu(group_0, group_1).pvalue,
            "AUC": max(auc, 1 - auc),  # direction-free: 0.5 = useless, 1.0 = perfect separation
        }
    )

bivariate_numerical = pd.DataFrame(bivariate_numerical).set_index("variable").sort_values("AUC", ascending=False)
print(bivariate_numerical.round(4))

Correlations and tests compress the relationship into one number and assume a particular form (linear, or at least monotone). Plotting the **target rate across bins** of the feature reveals the actual shape.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

# age: equal-width bins show the shape in the tails
age_bins = pd.cut(train["age"], bins=[16, 25, 30, 35, 40, 45, 50, 55, 60, 100])
age_rate = train.groupby(age_bins, observed=True)["subscribed"].agg(["mean", "size"])
axes[0].bar(age_rate.index.astype(str), age_rate["mean"], color="steelblue")
axes[0].set_title("Subscription Rate by Age", fontsize=14, fontweight="bold")
axes[0].tick_params(axis="x", rotation=45)

campaign_rate = train.groupby(train["campaign"].clip(upper=10))["subscribed"].mean()
axes[1].bar(campaign_rate.index.astype(str), campaign_rate.values, color="steelblue")
axes[1].set_title("Subscription Rate by Number of Contacts (10 = 10 or more)", fontsize=14, fontweight="bold")

euribor_bins = pd.qcut(train["euribor3m"], q=8, duplicates="drop")
euribor_rate = train.groupby(euribor_bins, observed=True)["subscribed"].mean()
axes[2].bar(euribor_rate.index.astype(str), euribor_rate.values, color="steelblue")
axes[2].set_title("Subscription Rate by Euribor 3M (quantile bins)", fontsize=14, fontweight="bold")
axes[2].tick_params(axis="x", rotation=45)

for ax in axes:
    ax.axhline(train["subscribed"].mean(), color="red", linestyle="--", label="overall rate")
    ax.set_ylabel("Subscription rate", fontsize=12)
    ax.legend()

plt.tight_layout()
plt.show()

print("Number of training observations per age bin:")
print(age_rate["size"].to_string())

## 7.2 Categorical Features vs. Target

For every category we compute the subscription rate with a **95% confidence interval** (Wilson interval for a proportion) — essential, because a rate computed from 30 observations is far less reliable than one computed from 10,000.

In [ ]:
from statsmodels.stats.proportion import proportion_confint


def target_rate_table(data, column, target="subscribed"):
    """Target rate by category with Wilson 95% confidence intervals."""
    table = data.groupby(column, observed=True)[target].agg(n="size", positives="sum")
    table["rate"] = table["positives"] / table["n"]
    table["ci_low"], table["ci_high"] = proportion_confint(table["positives"], table["n"], alpha=0.05, method="wilson")
    return table


fig, axes = plt.subplots(2, 3, figsize=(22, 11))
for ax, column in zip(axes.ravel(), ["job", "education", "default", "contact", "month", "poutcome"]):
    table = target_rate_table(train, column)
    if not train[column].cat.ordered:
        table = table.sort_values("rate")
    errors = [table["rate"] - table["ci_low"], table["ci_high"] - table["rate"]]
    ax.barh(table.index.astype(str), table["rate"], xerr=errors, color="steelblue", capsize=3)
    ax.axvline(train["subscribed"].mean(), color="red", linestyle="--", label="overall rate")
    ax.set_title(f"Subscription Rate by {column}", fontsize=13, fontweight="bold")
    ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print(target_rate_table(train, "poutcome").round(4))

## 7.3 Ranking the Features: Cramér's V and Information Value

Two measures that put all features on a common scale.

**Cramér's V** rescales the χ² statistic of a contingency table to the interval [0, 1]:

$$V = \sqrt{\frac{\chi^2 / n}{\min(r - 1, c - 1)}}$$

**Weight of Evidence (WoE) and Information Value (IV)** are the standard tools of **credit scoring**. For each category (or bin) $i$ of a feature:

$$\text{WoE}_i = \ln\frac{\text{share of all events falling into } i}{\text{share of all non-events falling into } i}, \qquad \text{IV} = \sum_i \big(\text{share of events}_i - \text{share of non-events}_i\big) \cdot \text{WoE}_i$$

A common rule of thumb for the IV: < 0.02 not predictive, 0.02–0.1 weak, 0.1–0.3 medium, 0.3–0.5 strong, **> 0.5 suspicious — check for leakage**. Numerical variables are binned first (here: into up to 10 quantile bins; missing values form their own bin).

Both measures look at one feature at a time — they are an EDA ranking, not a final feature selection. Model-based selection methods (RFE, sequential selection, L1) are in `core_ml_techniques.ipynb`, Section 4.

In [ ]:
def cramers_v(x, y):
    """Cramér's V between two categorical variables."""
    contingency = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(contingency)[0]
    return np.sqrt(chi2 / contingency.to_numpy().sum() / (min(contingency.shape) - 1))


def information_value(feature, target, n_bins=10):
    """Information Value of a feature with respect to a binary target."""
    if pd.api.types.is_numeric_dtype(feature) and feature.nunique() > n_bins:
        feature = pd.qcut(feature, q=n_bins, duplicates="drop")
    feature = feature.astype(object).where(feature.notna(), "MISSING")
    counts = pd.crosstab(feature, target)
    # 0.5 is added to every cell so that empty cells do not produce log(0)
    share_events = (counts[1] + 0.5) / (counts[1] + 0.5).sum()
    share_non_events = (counts[0] + 0.5) / (counts[0] + 0.5).sum()
    woe = np.log(share_events / share_non_events)
    return ((share_events - share_non_events) * woe).sum()


feature_columns = numerical_columns + categorical_columns + ["contacted_before"]
ranking = pd.DataFrame(
    {
        "IV": {column: information_value(train[column], train["subscribed"]) for column in feature_columns},
        "Cramér's V (categorical only)": {
            column: cramers_v(train[column], train["subscribed"]) for column in categorical_columns
        },
    }
).sort_values("IV", ascending=False)

print(ranking.round(4))

**What to notice:** the top of the ranking is occupied by the macro variables and by `month` — the variables that identify *when* the call was made — with IVs close to the "suspicious" threshold. The client-level variables (`contact`, `poutcome`, `job`, `age`) are far behind. Together with Section 3.5 this tells us that the strongest "signal" in this dataset is calendar time (the change of the bank's strategy and of the economy), which is exactly the kind of signal that may not transfer to a future campaign. Note also that `cons_price_idx` and `cons_conf_idx` have *identical* IVs: both take one value per month, so after binning they carry the same information.

# 8. Multivariate Analysis

Features do not act in isolation. Two questions matter most before modelling: **are the features redundant** (multicollinearity), and **does the effect of one feature depend on another** (interactions)?

## 8.1 Correlations Between Numerical Features

Pearson's coefficient measures *linear* association, Spearman's rank correlation measures *monotone* association and is robust to outliers and skewness. Large differences between the two point to non-linearity or to outliers.

In [ ]:
corr_columns = [c for c in numerical_columns if c != "pdays"] + ["contacted_before"]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
for ax, method in zip(axes, ["pearson", "spearman"]):
    corr = train[corr_columns].corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool))  # hide the redundant upper triangle
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax, cbar_kws={"shrink": 0.8})
    ax.set_title(f"{method.capitalize()} Correlation (training set)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 8.2 Multicollinearity: Variance Inflation Factor

Pairwise correlations can miss a variable that is a combination of *several* others. The **Variance Inflation Factor** regresses each feature on all the others:

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

A VIF of 10 means that the variance of the estimated coefficient is 10 times larger than it would be without collinearity (standard errors larger by a factor of $\sqrt{10} \approx 3.2$). Rules of thumb: VIF > 5 deserves attention, VIF > 10 is serious.

For EDA, the VIF answers a simple question: **how many distinct pieces of information do these columns really carry?** Multicollinearity does not, by itself, harm predictions, but it makes the coefficients of linear models unstable and uninterpretable (their signs may flip, their standard errors explode) — you will see this, and its cure (regularisation), in `linear_and_logistic_regression.ipynb`.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


def vif_table(data, columns):
    """VIF of every column (a constant is added, as required for a meaningful VIF)."""
    design = sm.add_constant(data[columns].astype(float))
    return pd.Series(
        [variance_inflation_factor(design.values, i) for i in range(1, design.shape[1])], index=columns, name="VIF"
    )


macro_columns = ["emp_var_rate", "cons_price_idx", "cons_conf_idx", "euribor3m", "nr_employed"]
all_columns = ["age", "campaign", "previous"] + macro_columns

print("VIF with all five macro variables:")
print(vif_table(train, all_columns).round(1))

reduced_columns = ["age", "campaign", "previous", "euribor3m", "cons_conf_idx"]
print("\nVIF after keeping only euribor3m and cons_conf_idx:")
print(vif_table(train, reduced_columns).round(1))

## 8.3 Interactions

An interaction means that the effect of one feature depends on the level of another. A pivot table of target rates is a simple way to look for them.

In [ ]:
interaction = train.pivot_table(index="contact", columns="poutcome", values="subscribed", aggfunc="mean", observed=True)
counts = train.pivot_table(index="contact", columns="poutcome", values="subscribed", aggfunc="size", observed=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
sns.heatmap(interaction, annot=True, fmt=".3f", cmap="YlGnBu", ax=axes[0])
axes[0].set_title("Subscription Rate: contact x poutcome", fontsize=14, fontweight="bold")
sns.heatmap(counts, annot=True, fmt=",d", cmap="Greys", ax=axes[1])
axes[1].set_title("Number of Observations", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

**What to notice:** for clients without a previous campaign, calling a mobile phone works better than a landline; for clients whose previous campaign was a *success*, the ordering flips. Before reading anything into that flip, look at the right panel: it rests on a handful of landline calls. Interactions found in EDA are hypotheses; the ones worth adding to a model are those that are large **and** supported by enough observations.

## 8.4 A Low-Dimensional View with PCA

Unsupervised techniques are useful in EDA as well. Principal Component Analysis projects the (standardised) numerical features onto the directions of largest variance. Two questions: how many dimensions does the numerical information really have, and do the classes separate in that space?

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pca_columns = ["age", "campaign", "previous"] + macro_columns
X_pca = StandardScaler().fit_transform(train[pca_columns])
pca = PCA().fit(X_pca)
components = pca.transform(X_pca)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].bar(range(1, len(pca_columns) + 1), pca.explained_variance_ratio_, label="individual")
axes[0].plot(
    range(1, len(pca_columns) + 1), np.cumsum(pca.explained_variance_ratio_), "o-", color="red", label="cumulative"
)
axes[0].set_xlabel("Principal component", fontsize=12)
axes[0].set_ylabel("Share of variance explained", fontsize=12)
axes[0].set_title("PCA: Explained Variance", fontsize=14, fontweight="bold")
axes[0].legend()

sample = np.random.default_rng(42).choice(len(train), size=5000, replace=False)
scatter = axes[1].scatter(
    components[sample, 0],
    components[sample, 1],
    c=train["subscribed"].to_numpy()[sample],
    cmap="coolwarm",
    alpha=0.5,
    s=12,
)
axes[1].set_xlabel("PC 1", fontsize=12)
axes[1].set_ylabel("PC 2", fontsize=12)
axes[1].set_title("First Two Principal Components (random sample of 5,000)", fontsize=14, fontweight="bold")
axes[1].legend(*scatter.legend_elements(), title="subscribed")

plt.tight_layout()
plt.show()

loadings = pd.DataFrame(pca.components_[:2].T, index=pca_columns, columns=["PC 1", "PC 2"])
print("Loadings of the first two components:")
print(loadings.round(2))
print(
    f"\nVariance explained by PC 1: {pca.explained_variance_ratio_[0]:.1%}, by PC 1 + PC 2: {pca.explained_variance_ratio_[:2].sum():.1%}"
)

**What to notice:** the first component loads almost exclusively on the five macro variables — they form **one** dimension, confirming the VIF analysis — while the second component is essentially `age`. The two classes overlap heavily in this plane: the numerical variables alone separate subscribers from non-subscribers only weakly, consistent with the modest univariate AUCs of Section 7.1.

# 9. Is the Past Representative of the Future? Feature Stability

Section 5 showed that the training and test periods differ in the subscription rate and in the interest-rate environment. Which features are affected, and how strongly? Credit-risk teams answer this with the **Population Stability Index (PSI)**, which compares the distribution of a feature in a reference sample (here: the training period) with a new sample (the test period), bin by bin:

$$\text{PSI} = \sum_i (a_i - e_i) \ln\frac{a_i}{e_i},$$

where $e_i$ and $a_i$ are the shares of the reference and of the new sample falling into bin $i$ (bins: deciles of the reference sample for numerical features, categories for categorical ones, missing values as their own bin). Rule of thumb: **< 0.1** stable, **0.1–0.25** moderate shift, **> 0.25** major shift.

The PSI uses the **features only, never the target** — exactly as in production, where the features of the clients to be scored are known long before their outcomes. Looking at the test-period features therefore does not leak any information about the target.

In [ ]:
def population_stability_index(reference, new, n_bins=10):
    """PSI of one feature between a reference sample and a new sample."""
    if pd.api.types.is_numeric_dtype(reference) and reference.nunique() > n_bins:
        # Bin edges come from the reference sample only; the outer bins are open-ended
        inner_edges = np.unique(reference.quantile(np.linspace(0, 1, n_bins + 1)).to_numpy()[1:-1])
        edges = np.concatenate([[-np.inf], inner_edges, [np.inf]])
        reference, new = pd.cut(reference, edges), pd.cut(new, edges)
    reference = reference.astype(object).where(reference.notna(), "MISSING")
    new = new.astype(object).where(new.notna(), "MISSING")
    counts = pd.concat([reference.value_counts(), new.value_counts()], axis=1, keys=["reference", "new"]).fillna(0)
    # 0.5 is added to every cell so that a bin empty in one sample does not produce log(0)
    shares = (counts + 0.5) / (counts + 0.5).sum()
    return ((shares["new"] - shares["reference"]) * np.log(shares["new"] / shares["reference"])).sum()


stability = pd.Series(
    {column: population_stability_index(train[column], test[column]) for column in feature_columns}, name="PSI"
).sort_values(ascending=False)
stability = stability.to_frame()
stability["verdict"] = pd.cut(
    stability["PSI"], bins=[-np.inf, 0.1, 0.25, np.inf], labels=["stable", "moderate shift", "major shift"]
)
print(stability.round(3))

**What to notice:**

- The five macro variables have PSIs between 8 and 13 — 30 to 50 times the "major shift" threshold. This is not a gradual drift: the Euribor values of the test period (0.6–1.3%) lie at or below the lowest value seen in training (1.3–5.0%). A model that relies on these features has to **extrapolate**. `month` shifts strongly as well, because the two periods cover different months.
- The variables describing **previous campaigns and the contact channel** also shift a lot, although they describe the client. In the training period 7% of the contacted clients had been contacted before, in the test period 41%; `poutcome == "success"` rises from 0.5% to 15%, and the share of mobile phones from 57% to 88%. The bank changed *whom* it called: later campaigns focused on clients it already knew, reached on their mobile phones.
- `age` and `default` show a moderate shift (more clients over 60, far fewer `"unknown"` default statuses); the clients' own characteristics — `education`, `marital`, `housing`, `loan` — and `campaign` and `day_of_week` are stable.
- Conclusion: the test period differs not only in the economy but in the **composition of the clients**. This is why the out-of-time split of Section 5 is the honest test, and why a deployed model needs its input distributions monitored (`core_ml_techniques.ipynb`, Section 8).

# 10. Feature Engineering

Feature engineering turns what we learned during EDA into variables a model can use. Two principles:

1. **Write transformations as functions**, so that exactly the same code is applied to the training set, the test set and — later — to new data in production.
2. Distinguish **stateless** transformations (a logarithm, a flag, a ratio — they need nothing from the data and can be applied anywhere) from **stateful** ones (imputation with the median, scaling, one-hot or target encoding, quantile bins — they *learn* parameters and must be fitted on the training set only, inside a scikit-learn `Pipeline`).

Here we write only the stateless features that follow directly from our EDA findings. The stateful transformations are not applied by hand: they become steps of the modelling pipeline (`core_ml_techniques.ipynb`, Sections 1–2, and the Kaggle League starter).

## 10.1 Stateless Features

In [ ]:
def add_features(data):
    """Stateless feature engineering - safe to apply to any subset of the data."""
    data = data.copy()

    # Counts with a long right tail -> logarithm (log1p = log(1 + x) handles zeros)
    data["campaign_log"] = np.log1p(data["campaign"])

    # pdays is observed for very few clients: keep the information as an ordered 'recency' category
    data["recency"] = pd.cut(
        data["pdays"], bins=[-1, 3, 7, 30], labels=["0-3 days", "4-7 days", "8+ days"]
    ).cat.add_categories("no recent contact")
    data["recency"] = data["recency"].fillna("no recent contact")

    # The age effect is U-shaped (young and old clients subscribe more often) -> bins instead of a linear term
    data["age_group"] = pd.cut(
        data["age"], bins=[0, 25, 35, 45, 55, 65, 120], labels=["<=25", "26-35", "36-45", "46-55", "56-65", "65+"]
    )

    # Section 3.5: the Euribor collapse splits the sample into two interest-rate regimes
    data["low_rate_regime"] = (data["euribor3m"] < 2.0).astype(int)

    # Flags derived from the 'unknown' analysis
    data["default_unknown"] = (data["default"] == "unknown").astype(int)

    return data


train_fe = add_features(train)
test_fe = add_features(test)

new_columns = ["campaign_log", "recency", "age_group", "low_rate_regime", "default_unknown"]
print(train_fe[new_columns].head())
print("\nSubscription rate by the new categorical features (training set):")
for column in ["recency", "age_group"]:
    print(f"\n{target_rate_table(train_fe, column)[['n', 'rate']].round(4)}")

## 10.2 Saving the Prepared Data

Save the cleaned data so that the modelling notebooks can start from it. The format matters:

| Format | Keeps dtypes (categories, dates)? | Size / speed | Readable by other tools |
|---|---|---|---|
| CSV | no — everything is re-parsed from text, ordered categories are lost | large / slow | everything |
| **Parquet** | yes | small / fast, columnar | Python, R, Spark, DuckDB, ... — the de-facto standard |
| Pickle | yes | fast | Python only; unsafe to load from untrusted sources; breaks between library versions |

Parquet is the recommended choice (`df.to_parquet(...)`); it requires the `pyarrow` package, which is not part of the course environment (`uv add pyarrow` — see `SETUP.md`). Below we write to a temporary directory and demonstrate what CSV does to the data types.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp_dir:
    csv_path = Path(tmp_dir) / "bank_train.csv.gz"
    train_fe.to_csv(csv_path, index=False)
    reloaded = pd.read_csv(csv_path)
    print(f"Size of the compressed CSV: {csv_path.stat().st_size / 1024:.0f} KB")

comparison = pd.DataFrame(
    {"before saving": train_fe.dtypes.astype(str), "after CSV round-trip": reloaded.dtypes.astype(str)}
)
print("\nColumns whose data type changed:")
print(comparison[comparison["before saving"] != comparison["after CSV round-trip"]])

# 11. Key Takeaways and Best Practices

## What We Learned about This Dataset

- `info()` showed no missing values; in reality several columns had them, disguised as `"unknown"` and `999`.
- A consistency check revealed that the documentation of `pdays` is incomplete.
- The rows are ordered in time; the bank's behaviour and the economy changed radically during the sample.
- `duration` is a textbook case of target leakage.
- The five macro variables are one signal, not five — and they are the features that shift most between the training and the test period (Section 9).
- The bank also changed whom it called: previously contacted clients and mobile phones dominate the later period.

## From EDA to Modelling Decisions

The EDA ends with decisions that the modelling stage inherits:

| Decision | Reason |
|---|---|
| Drop `duration` | target leakage (Section 4) |
| Keep `"unknown"` as a category; derive `contacted_before` from `previous` | informative missingness, inconsistent `pdays` (Section 3) |
| Validate out of time — hold-out on the latest contacts, time-ordered cross-validation | the data is ordered in time and the world changed (Sections 3.5, 5, 9) |
| Keep at most one or two macro variables, or let regularisation deal with them | one signal, extreme VIF (Section 8.2) |
| Treat the macro and calendar variables with suspicion; do not expect the previous-campaign variables to behave in the future as they did in training | the strongest univariate signal is calendar time, and it is the least stable one; the client mix changed (Sections 7.3, 9) |
| Log-transform the counts, bin `age`, keep the rare categories under control | skewness, U-shape, rare categories (Sections 6, 7) |
| Report ROC AUC and PR AUC, not accuracy | ~11% positive class (Section 2) |

## Best Practices

### 1. **Look at the raw file before `read_csv`**
Separator, decimal mark, encoding, header, how missing values are written.

### 2. **Never trust `info()` on missing values**
Search for sentinels (`unknown`, `?`, `-1`, `999`, `0`) and check whether missingness is informative before imputing anything.

### 3. **Validate against the business logic**
Cross-tabulate variables that must agree with each other; every violation is either an error or a lesson about the data.

### 4. **Reconstruct and plot the time dimension**
Even "cross-sectional" data is usually collected over time. Plot the target rate and the key features against time before deciding how to split.

### 5. **Hunt for leakage**
For every feature ask whether it would be known at prediction time; distrust features with implausibly high univariate power.

### 6. **Split first, explore later**
Everything that touches the target — plots, statistics, encodings, feature rankings — is done on the training set.

### 7. **Effect sizes over p-values**
With $n$ in the tens of thousands everything is significant. Report confidence intervals, Cramér's V, IV, AUC.

### 8. **Plot the shape of relationships**
A correlation coefficient hides U-shapes and thresholds; target rate by bins does not.

### 9. **Measure redundancy and stability**
The VIF tells you how many distinct signals a group of features carries; the PSI tells you which features will look different in the period you predict.

### 10. **Write transformations as functions, keep stateful steps in a Pipeline**
The same code must run on training data, test data and new data.

## Common Pitfalls to Avoid

1. **Imputing sentinels with the mean** → the "999 days" client becomes the average client
2. **Deleting outliers without a business reason** → the tail is often where the information is
3. **Random splits on time-ordered data** → optimistic evaluation, unpleasant surprise in production
4. **Encoding / scaling before the split** → statistics leak from the test set
5. **Trusting the most predictive feature blindly** → in this dataset it mostly encodes calendar time, which is the least stable signal
6. **Reading "significant" as "important"** → check the effect size
7. **Saving the cleaned data as CSV** → categories, order and dates are lost on reload

## Further Reading

- [pandas User Guide](https://pandas.pydata.org/docs/user_guide/index.html) — especially *Working with missing data*, *Categorical data*, *Group by*
- [seaborn tutorial](https://seaborn.pydata.org/tutorial.html) — the visual vocabulary of EDA
- [scikit-learn: Common pitfalls and recommended practices](https://scikit-learn.org/stable/common_pitfalls.html) — data leakage, pipelines, randomness
- Siddiqi, N. (2017). *Intelligent Credit Scoring* — WoE, IV and the practice of building scorecards
- James, G., Witten, D., Hastie, T., Tibshirani, R. (2021). *An Introduction to Statistical Learning*, Chapters 3–4

# 12. Homework Assignment

## EDA for Credit Card Default

### Objective
Apply the complete workflow of this notebook to a new dataset: ingest, clean, validate, explore, engineer features, and turn your findings into a modelling plan.

### Dataset
Use the **Default of Credit Card Clients** dataset (UCI Machine Learning Repository, 30,000 clients of a Taiwanese bank, 23 features — credit limit, demographics, six months of repayment status, bill and payment amounts). It ships with the repository:

```python
import pandas as pd

df = pd.read_csv("../data/credit_card_default.csv.gz")
```

The meaning of every column is described on the [UCI page](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients) and in [`data/README.md`](../data/README.md). Target: `default_next_month` (1 = default, ≈ 22%).

### Tasks

#### Part 1: Ingestion and Cleaning
1. Load the data, rename the columns to readable names, fix the data types
2. Check for duplicates and for identifier columns that must not be used as features
3. Compare the value counts of `EDUCATION`, `MARRIAGE` and the `PAY_*` columns with the documentation — find the undocumented codes and decide how to treat them (justify your decision)
4. Reconstruct the time structure of the `PAY_*`, `BILL_AMT*` and `PAY_AMT*` columns (which month is which?)

#### Part 2: Leakage and Splitting
1. Discuss which columns could leak the target, and check the univariate AUC of every feature
2. Split the data into training (70%) and test (30%) sets with stratification (the data has no time order — explain why a random split is acceptable here, in contrast to this notebook)

#### Part 3: Exploratory Analysis (training set only)
1. Univariate analysis of all variables: distributions, skewness, outliers, rare categories
2. Bivariate analysis: default rate by category with confidence intervals; default rate by bins of the numerical variables; the shape of the relationship between credit limit and default
3. Compute the Information Value of every feature and rank them
4. Correlation heatmaps (Pearson and Spearman) of the numerical variables; VIF of the bill amounts — what do you conclude?
5. Look for at least one interaction (e.g. repayment status × credit limit)

#### Part 4: Feature Engineering
1. Write an `add_features` function with at least five new features, e.g. utilisation ratio (bill / limit), payment-to-bill ratio, number of months in arrears, trend of the bill amounts, maximum delay
2. Compute the Information Value of the new features — do they add anything beyond the raw columns?

#### Part 5: Stability and the Modelling Plan
1. Compute the PSI of every feature between your training and test sets and compare the values with Section 9 of this notebook — explain the difference
2. Write a "From EDA to modelling decisions" table like the one in Section 11: which features to drop, transform or merge, which validation scheme and which metric to use — each decision with a reason from your analysis
3. Save the prepared training and test sets and verify that the data types survive the round trip

#### Part 6: Report
1. Summarise your findings in a short report (markdown cells): the five most important insights about the data, and the three variables most strongly related to default — with the direction and the size of the relationship (target rates, IV)
2. **Bonus**: Discuss which of the features would be problematic to use in a real credit scoring model (regulation, fairness), and whether the `PAY_*` codes are a form of leakage